# 🛠️ Stage 3 — Preprocessing & Feature Engineering
### 🏦 Loan Default Predictor — Home Credit Dataset
---
**Clean the data, handle missing values, encode categories, scale features, and create new features.**

> **Sections 4 & 5**

```
Progress: ███░░  Stage 3 of 5
```

← [Stage 2](stage_02_eda.ipynb)  |  [Stage 4](stage_04_models_and_evaluation.ipynb) →

---
### 📋 What you will do in this stage:
- Select the most useful **features** from the dataset
- Handle **missing values** with median and mode imputation
- Fix **anomalous values** (the 365243 employment anomaly)
- Apply **One-Hot Encoding** to categorical variables
- Split data into **train / test sets** (stratified)
- Apply **StandardScaler** to normalize numerical features
- Engineer **new features** (ratios, averages) to boost model performance

⏱️ *Estimated time: 40–60 minutes*

---
> ⚠️ **Prerequisite:** This notebook depends on **Stages 1 & 2 (EDA insights guide our preprocessing decisions)**.  
> Run the previous stage(s) first, **or** run the cell below to reload saved objects.


### ⚙️ Reload Cell
Run this if you are starting fresh without completing Stages 1 & 2 first.


In [ ]:
# ── RELOAD: Run this only if you didn't just complete Stages 1 & 2 ──
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 12

df = pd.read_csv("data/raw/application_train.csv")
df["AGE_YEARS"] = (-df["DAYS_BIRTH"] / 365).round(1)

print(f"✅ Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

---
## Section 4 — Data Preprocessing

Raw data is messy. Before feeding it to a machine learning model, we need to:
1. **Handle missing values** — models can't work with NaN
2. **Encode categorical variables** — models work with numbers, not text
3. **Scale numerical features** — put all numbers on the same scale
4. **Split into train and test sets** — so we can evaluate honestly

> 💡 **The golden rule:** Any transformation learned on training data  
> must be *applied* (not re-learned) on test data.  
> Example: if the mean income from training is $50,000, we use that to fill missing  
> values in the test set — **not** the test set's own mean.


### 4.1 — Select Features

With 122 columns, we'll start with a focused set of the most important features.  
This keeps the notebook manageable and teaches the core concepts clearly.  
In a real project you'd include more features after this foundation is solid.


In [ ]:
# Selected features — a mix of financial, demographic, and credit score data
FEATURES = [
    # External credit scores (usually most predictive)
    "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3",
    # Financial info
    "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE",
    # Days-based (we'll convert to years during feature engineering)
    "DAYS_BIRTH", "DAYS_EMPLOYED",
    # Demographics
    "CODE_GENDER", "NAME_EDUCATION_TYPE", "NAME_INCOME_TYPE",
    "NAME_FAMILY_STATUS", "NAME_HOUSING_TYPE", "OCCUPATION_TYPE",
    # Flags
    "FLAG_OWN_CAR", "FLAG_OWN_REALTY",
    # Other useful features
    "CNT_CHILDREN", "CNT_FAM_MEMBERS", "REGION_RATING_CLIENT",
    "REG_CITY_NOT_WORK_CITY", "DEF_30_CNT_SOCIAL_CIRCLE"
]

TARGET = "TARGET"

# Create working copies
X = df[FEATURES].copy()
y = df[TARGET].copy()

print(f"Features selected : {len(FEATURES)}")
print(f"Training samples  : {len(X):,}")

### 4.2 — Handle Missing Values

**Strategy by column type:**
- **Numerical columns** → fill with the **median** (median is less affected by outliers than mean)
- **Categorical columns** → fill with `"Unknown"` (creates a new category for missing)

> 🔬 **Mini-example — Why median over mean for imputation?**  
> Income = [20k, 25k, 30k, 28k, 500k]  
> Mean = 120.6k → pulled high by the outlier  
> Median = 28k → much more representative of a typical person


In [ ]:
# Identify column types
categorical_cols = X.select_dtypes(include="object").columns.tolist()
numerical_cols   = X.select_dtypes(exclude="object").columns.tolist()

print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")
print(f"
Numerical columns ({len(numerical_cols)}): {numerical_cols}...")

In [ ]:
# Fill missing values in NUMERICAL columns with the median
for col in numerical_cols:
    median_val = X[col].median()
    X[col] = X[col].fillna(median_val)

In [ ]:
# Fill missing values in CATEGORICAL columns with "Unknown"
for col in categorical_cols:
    X[col] = X[col].fillna("Unknown")

# Confirm no missing values remain
print(f"Missing values remaining: {X.isnull().sum().sum()}")

### 4.3 — Fix Anomalous Values

Some values in `DAYS_EMPLOYED` are `365243` — this is a placeholder for unemployed people.  
We'll replace it with 0 before converting to years.


In [ ]:
# Check DAYS_EMPLOYED anomaly
print(f"Max DAYS_EMPLOYED: {X['DAYS_EMPLOYED'].max():,}")
print(f"Count of 365243 values: {(X['DAYS_EMPLOYED'] == 365243).sum():,}")

In [ ]:
# Replace the anomalous value with 0 (not employed)
X["DAYS_EMPLOYED"] = X["DAYS_EMPLOYED"].replace(365243, 0)

print(f"Max DAYS_EMPLOYED after fix: {X['DAYS_EMPLOYED'].max():,}")

### 4.4 — Encode Categorical Variables

Machine learning models work with **numbers**, not text.  
We convert text categories to numbers using **One-Hot Encoding**.

> 💡 **What is One-Hot Encoding?**  
> If `CODE_GENDER` has values `M` and `F`, we create two new binary columns:  
> `CODE_GENDER_M` (1 if Male, 0 otherwise) and `CODE_GENDER_F` (1 if Female, 0 otherwise)  
> We then drop one to avoid redundancy (this is called the "dummy variable trap").


In [ ]:
# 🔬 Mini-example of One-Hot Encoding
import pandas as pd

example = pd.DataFrame({"gender": ["M", "F", "M", "F"]})
encoded = pd.get_dummies(example, drop_first=True)
print("Before encoding:")
print(example)
print("
After encoding:")
print(encoded)

In [ ]:
# Apply One-Hot Encoding to our actual data
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

print(f"Shape before encoding: {len(FEATURES)} features")
print(f"Shape after encoding : {X.shape[1]} features")
print(f"
Sample new column names: {list(X.columns[-5:])}")

### 4.5 — Train / Test Split

We split the data **before** scaling. This prevents **data leakage** — where information  
from the test set accidentally influences how we prepare the training data.

> 💡 **What is data leakage?**  
> Imagine studying for an exam using the answer key. You'd score 100%,  
> but you haven't truly learned anything. Data leakage is the same:  
> the model learns from information it shouldn't have access to yet.

We use `stratify=y` to ensure both splits have the same proportion of defaults (~8.1%).


In [ ]:
# Split: 80% train, 20% test — stratified to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y       # <-- ensures similar default rates in both splits
)

print(f"Training samples : {len(X_train):,}")
print(f"Test samples     : {len(X_test):,}")
print(f"
Default rate — train : {y_train.mean():.3f}")
print(f"Default rate — test  : {y_test.mean():.3f}")

### 4.6 — Feature Scaling

Some algorithms (like Logistic Regression) are sensitive to feature scale.  
A feature with values in the millions (income) will dominate a feature with values 0–1 (external scores).  

We use `StandardScaler` which transforms each feature to have:
- **Mean = 0**
- **Standard deviation = 1**

> 🔬 **Mini-example — Why scaling matters:**  
> Without scaling: income (50,000) vs external score (0.6) → model focuses on income  
> With scaling: both become values like 0.3 or -1.2 → model treats them equally


In [ ]:
# Initialize the scaler
scaler = StandardScaler()

# FIT on training data only — learn the mean and std from training set
X_train_scaled = scaler.fit_transform(X_train)

# TRANSFORM test data using training set's mean and std
X_test_scaled  = scaler.transform(X_test)

print(f"Scaling complete.")
print(f"Training set shape: {X_train_scaled.shape}")
print(f"
Example — first feature before scaling: mean={X_train.iloc[:,0].mean():.2f}")
print(f"Example — first feature after scaling:  mean≈{X_train_scaled[:,0].mean():.4f}")

---
## Section 5 — Feature Engineering

Feature engineering means **creating new, more informative features** from existing ones.  
Good features often matter more than which model you choose.

> 💡 **Analogy:** A doctor doesn't just look at your blood pressure number alone.  
> They consider your age, weight, family history — combinations that reveal risk.  
> Feature engineering lets us build those combinations for the model.


We'll add engineered features to our original dataset **before** the train/test split  
and preprocessing. Let me show the engineering step here as a separate demonstration  
that you can incorporate into the pipeline.


In [ ]:
# Create a fresh copy to demonstrate feature engineering
df_fe = df[FEATURES + [TARGET]].copy()

# Fix the employment anomaly first
df_fe["DAYS_EMPLOYED"] = df_fe["DAYS_EMPLOYED"].replace(365243, 0)

In [ ]:
# Feature 1: Age in years (readable)
df_fe["AGE_YEARS"] = (-df_fe["DAYS_BIRTH"] / 365).round(1)

# Feature 2: Employment length in years
df_fe["EMPLOYED_YEARS"] = (-df_fe["DAYS_EMPLOYED"] / 365).round(1)

In [ ]:
# Feature 3: Credit-to-Income Ratio
# "How many years of income does this loan represent?"
# High ratio = risky borrower
df_fe["CREDIT_INCOME_RATIO"] = df_fe["AMT_CREDIT"] / (df_fe["AMT_INCOME_TOTAL"] + 1)

In [ ]:
# Feature 4: Annuity-to-Income Ratio
# "What % of monthly income goes to loan repayment?"
# Similar to Debt-to-Income (DTI) ratio used by banks
df_fe["ANNUITY_INCOME_RATIO"] = df_fe["AMT_ANNUITY"] / (df_fe["AMT_INCOME_TOTAL"] + 1)

In [ ]:
# Feature 5: Average External Score across all 3 sources
df_fe["EXT_SOURCE_MEAN"] = df_fe[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].mean(axis=1)

In [ ]:
# Preview the new features
new_features = ["AGE_YEARS", "EMPLOYED_YEARS", "CREDIT_INCOME_RATIO",
                "ANNUITY_INCOME_RATIO", "EXT_SOURCE_MEAN"]
print("New engineered features — sample values:")
df_fe[new_features].describe().round(3)

In [ ]:
# Do the new features correlate with default?
print("Correlation of engineered features with TARGET:")
corr = df_fe[new_features + ["TARGET"]].corr()["TARGET"].drop("TARGET")
print(corr.sort_values().round(3).to_string())

**EXT_SOURCE_MEAN** has the strongest (negative) correlation with default.  
Lower external score = higher chance of default. This confirms our earlier finding.

For the model, we'll add these engineered features to our main dataset.


In [ ]:
# Add engineered features to X_train and X_test DataFrames
# (We do this BEFORE scaling, so they get scaled along with the rest)

def add_engineered_features(df_input):
    df_out = df_input.copy()
    df_out["CREDIT_INCOME_RATIO"]   = df_out.get("AMT_CREDIT",  0) / (df_out.get("AMT_INCOME_TOTAL", 1) + 1)
    df_out["ANNUITY_INCOME_RATIO"]  = df_out.get("AMT_ANNUITY", 0) / (df_out.get("AMT_INCOME_TOTAL", 1) + 1)
    return df_out

X_train = add_engineered_features(X_train)
X_test  = add_engineered_features(X_test)

# Re-scale after adding new features
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Feature count after engineering: {X_train.shape[1]}")

---
## ✅ Stage 3 Complete!

Great work! Here is a summary of what you accomplished:

- Select the most useful **features** from the dataset
- Handle **missing values** with median and mode imputation
- Fix **anomalous values** (the 365243 employment anomaly)
- Apply **One-Hot Encoding** to categorical variables
- Split data into **train / test sets** (stratified)
- Apply **StandardScaler** to normalize numerical features
- Engineer **new features** (ratios, averages) to boost model performance

⏱️ *Estimated time: 40–60 minutes*

---
### ➡️ Next: 🤖 Stage 4 — Handling Imbalance, Model Building & Evaluation
**Apply SMOTE, train Logistic Regression / Random Forest / XGBoost, and compare results.**

Open **`stage_04_models_and_evaluation.ipynb`** to continue.
